# Task 3: Architecture Comparison

Сравнение 3 архитектур:
1. **LLM без контекста** (baseline)
2. **Стандартный RAG** (top-K чанков)
3. **RAG + Reranker** (top-N -> top-K)

Метрики:
- Retrieval: Recall@K, MRR@K, Hit Rate@K
- Generation: BERTScore, ROUGE-L, BLEU

In [ ]:
import dotenv

In [ ]:
dotenv.load_dotenv('../../.env')

In [ ]:
import json
import os
import time
from pathlib import Path

import bert_score
import numpy as np
import pandas as pd
import sacrebleu
from datasets import load_dataset
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rouge_score import rouge_scorer

# Для ноутбуков используем текущую директорию
NOTEBOOK_DIR = Path.cwd()
ARTIFACTS_DIR = NOTEBOOK_DIR / "artifacts"

# Конфигурация
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "google/gemma-3-4b-it:free")
EMBED_MODEL = "bge-m3"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"

N_SAMPLES = 100
TOP_K = 5
TOP_N = 15

## Вспомогательные функции

In [ ]:
def get_llm():
    """Возвращает LLM через OpenRouter."""
    return ChatOpenAI(
        model=OPENROUTER_MODEL,
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        temperature=0,
    )

print(f"LLM: {OPENROUTER_MODEL}")
print(f"Embeddings: {EMBED_MODEL}")
print(f"Samples: {N_SAMPLES}")

## Retrieval метрики

In [ ]:
def compute_retrieval_metrics(retrieved_docs_list: list, ground_truth_contexts: list, top_k: int = TOP_K) -> dict:
    """Вычисляет retrieval метрики.
    
    Returns:
        dict с recall@k, mrr@k, hit_rate@k
    """
    recall_scores = []
    mrr_scores = []
    hit_scores = []

    for retrieved_docs, gt_context in zip(retrieved_docs_list, ground_truth_contexts):
        gt_normalized = gt_context.strip().lower()[:500]

        found_rank = None
        relevant_count = 0

        for rank, doc in enumerate(retrieved_docs[:top_k], 1):
            doc_normalized = doc.page_content.strip().lower()[:500]

            gt_words = set(gt_normalized.split())
            doc_words = set(doc_normalized.split())
            overlap = len(gt_words & doc_words)
            overlap_ratio = overlap / max(len(gt_words), 1)

            if overlap_ratio > 0.3:
                relevant_count += 1
                if found_rank is None:
                    found_rank = rank

        recall_scores.append(min(relevant_count, 1))
        mrr_scores.append(1.0 / found_rank if found_rank else 0.0)
        hit_scores.append(1 if relevant_count > 0 else 0)

    return {
        f"recall@{top_k}": np.mean(recall_scores),
        f"mrr@{top_k}": np.mean(mrr_scores),
        f"hit_rate@{top_k}": np.mean(hit_scores),
    }

## Generation метрики

In [ ]:
def compute_generation_metrics(predictions: list, references: list) -> dict:
    """Вычисляет метрики качества генерации."""
    valid_pairs = [(p, r) for p, r in zip(predictions, references) if p and r and not p.startswith("Error")]
    if not valid_pairs:
        return {"bertscore_f1": 0.0, "bertscore_precision": 0.0, "bertscore_recall": 0.0, "rouge_l": 0.0, "bleu": 0.0}

    preds, refs = zip(*valid_pairs)

    # BERTScore
    print("    Computing BERTScore...")
    P, R, F1 = bert_score.score(list(preds), list(refs), lang="en", verbose=False)
    bertscore_f1 = F1.mean().item()
    bertscore_precision = P.mean().item()
    bertscore_recall = R.mean().item()

    # ROUGE-L
    print("    Computing ROUGE-L...")
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_scores = [scorer.score(ref, pred)['rougeL'].fmeasure for pred, ref in zip(preds, refs)]
    rouge_l = np.mean(rouge_scores)

    # BLEU
    print("    Computing BLEU...")
    bleu_scores = []
    for pred, ref in zip(preds, refs):
        if pred.strip():
            try:
                bleu = sacrebleu.sentence_bleu(pred, [ref])
                bleu_scores.append(bleu.score)
            except Exception:
                bleu_scores.append(0.0)
        else:
            bleu_scores.append(0.0)
    bleu = np.mean(bleu_scores)

    return {
        "bertscore_f1": bertscore_f1,
        "bertscore_precision": bertscore_precision,
        "bertscore_recall": bertscore_recall,
        "rouge_l": rouge_l,
        "bleu": bleu,
    }

## Архитектура 1: No Context LLM

In [ ]:
class NoContextLLM:
    """Baseline: LLM без контекста."""

    def __init__(self):
        self.llm = get_llm()

    def run(self, question: str) -> dict:
        prompt = f"Question: {question}\n\nAnswer concisely and accurately:"

        t0 = time.perf_counter()
        try:
            response = self.llm.invoke(prompt)
            answer = response.content if hasattr(response, "content") else str(response)
        except Exception as e:
            answer = f"Error: {e}"
        latency = time.perf_counter() - t0

        return {"answer": answer, "latency": latency, "retrieved_docs": []}

## Архитектура 2: Standard RAG

In [ ]:
class RAGPipeline:
    """Стандартный RAG: retriever -> LLM."""

    def __init__(self, vector_store: FAISS, top_k: int = TOP_K):
        self.vector_store = vector_store
        self.top_k = top_k
        self.llm = get_llm()

    def run(self, question: str) -> dict:
        t0 = time.perf_counter()
        retriever = self.vector_store.as_retriever(search_kwargs={"k": self.top_k})
        docs = retriever.invoke(question)
        t_retrieve = time.perf_counter() - t0

        context = "\n\n".join(doc.page_content for doc in docs)

        prompt = f"""Context:
{context}

Question: {question}

Answer the question using only the context above. Be concise and accurate."""

        t1 = time.perf_counter()
        try:
            response = self.llm.invoke(prompt)
            answer = response.content if hasattr(response, "content") else str(response)
        except Exception as e:
            answer = f"Error: {e}"
        t_gen = time.perf_counter() - t1

        return {
            "answer": answer,
            "latency": t_retrieve + t_gen,
            "retrieval_time": t_retrieve,
            "generation_time": t_gen,
            "retrieved_docs": docs,
            "num_docs": len(docs),
        }

## Архитектура 3: RAG + Reranker

In [ ]:
class RAGWithReranker:
    """RAG + Reranker: retriever (top-N) -> reranker (top-K) -> LLM."""

    def __init__(self, vector_store: FAISS, top_n: int = TOP_N, top_k: int = TOP_K, reranker_model: str = RERANKER_MODEL):
        self.vector_store = vector_store
        self.top_n = top_n
        self.top_k = top_k
        self.llm = get_llm()
        print(f"Loading reranker: {reranker_model}")
        self.reranker = HuggingFaceCrossEncoder(model_name=reranker_model, model_kwargs={"device": "cpu"})

    def run(self, question: str) -> dict:
        t0 = time.perf_counter()
        retriever = self.vector_store.as_retriever(search_kwargs={"k": self.top_n})
        docs = retriever.invoke(question)
        t_retrieve = time.perf_counter() - t0

        t1 = time.perf_counter()
        if len(docs) > self.top_k:
            pairs = [[question, doc.page_content] for doc in docs]
            scores = self.reranker.score(pairs)
            ranked_indices = np.argsort(scores)[::-1][: self.top_k]
            docs = [docs[i] for i in ranked_indices]
        t_rerank = time.perf_counter() - t1

        context = "\n\n".join(doc.page_content for doc in docs)

        prompt = f"""Context:
{context}

Question: {question}

Answer the question using only the context above. Be concise and accurate."""

        t2 = time.perf_counter()
        try:
            response = self.llm.invoke(prompt)
            answer = response.content if hasattr(response, "content") else str(response)
        except Exception as e:
            answer = f"Error: {e}"
        t_gen = time.perf_counter() - t2

        return {
            "answer": answer,
            "latency": t_retrieve + t_rerank + t_gen,
            "retrieval_time": t_retrieve,
            "rerank_time": t_rerank,
            "generation_time": t_gen,
            "retrieved_docs": docs,
            "num_docs": len(docs),
        }

## Построение индекса из SQuAD

In [ ]:
def build_full_corpus_index(n_samples: int = N_SAMPLES) -> tuple:
    """Строит FAISS индекс из ПОЛНОГО датасета SQuAD."""
    print("Loading full SQuAD validation dataset...")
    full_dataset = load_dataset("rajpurkar/squad", split="validation")

    all_contexts = list(set(sample["context"] for sample in full_dataset))
    print(f"Found {len(all_contexts)} unique contexts")

    all_text = "\n\n".join(all_contexts)
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    chunks = splitter.split_text(all_text)

    docs = [Document(page_content=chunk) for chunk in chunks]
    embeddings = OllamaEmbeddings(model=EMBED_MODEL)

    print(f"Building FAISS index from {len(chunks)} chunks...")
    vector_store = FAISS.from_documents(docs, embeddings)

    samples = full_dataset.select(range(min(n_samples, len(full_dataset))))
    questions = [s["question"] for s in samples]
    ground_truth_contexts = [s["context"] for s in samples]
    references = [s["answers"]["text"][0] if s["answers"]["text"] else "" for s in samples]

    return vector_store, questions, ground_truth_contexts, references

# Построение индекса
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
vector_store, questions, ground_truth_contexts, references = build_full_corpus_index(N_SAMPLES)
print(f"Loaded {len(questions)} questions for evaluation")

## Функция оценки архитектуры

In [ ]:
def evaluate_architecture(pipeline, name: str, questions: list, ground_truth_contexts: list, references: list) -> dict:
    """Оценивает архитектуру и возвращает результаты."""
    print(f"\n--- Evaluating: {name} ---")

    predictions = []
    latencies = []
    all_retrieved_docs = []

    for i, question in enumerate(questions):
        print(f"  [{i + 1}/{len(questions)}] Processing...")
        result = pipeline.run(question)
        predictions.append(result["answer"])
        latencies.append(result["latency"])
        all_retrieved_docs.append(result.get("retrieved_docs", []))

    print("  Computing retrieval metrics...")
    retrieval_metrics = compute_retrieval_metrics(all_retrieved_docs, ground_truth_contexts, TOP_K)

    print("  Computing generation metrics...")
    gen_metrics = compute_generation_metrics(predictions, references)

    metrics = {
        "architecture": name,
        "recall@k": retrieval_metrics[f"recall@{TOP_K}"],
        "mrr@k": retrieval_metrics[f"mrr@{TOP_K}"],
        "hit_rate@k": retrieval_metrics[f"hit_rate@{TOP_K}"],
        "bertscore_f1": gen_metrics["bertscore_f1"],
        "bertscore_precision": gen_metrics["bertscore_precision"],
        "bertscore_recall": gen_metrics["bertscore_recall"],
        "rouge_l": gen_metrics["rouge_l"],
        "bleu": gen_metrics["bleu"],
        "avg_latency": np.mean(latencies),
    }

    print(f"  Retrieval: Recall@{TOP_K}={metrics['recall@k']:.4f}, MRR@{TOP_K}={metrics['mrr@k']:.4f}")
    print(f"  Generation: BERTScore={metrics['bertscore_f1']:.4f}, ROUGE-L={metrics['rouge_l']:.4f}")
    print(f"  Latency: {metrics['avg_latency']:.2f}s")

    return {"metrics": metrics, "predictions": predictions, "latencies": latencies}

## Инициализация архитектур

In [ ]:
no_context = NoContextLLM()
rag = RAGPipeline(vector_store, top_k=TOP_K)
rag_rerank = RAGWithReranker(vector_store, top_n=TOP_N, top_k=TOP_K)
print("All architectures initialized")

## Оценка архитектур

In [ ]:
results = {}

results["no_context"] = evaluate_architecture(no_context, "LLM (no context)", questions, ground_truth_contexts, references)
results["rag"] = evaluate_architecture(rag, "RAG", questions, ground_truth_contexts, references)
results["rag_rerank"] = evaluate_architecture(rag_rerank, "RAG + Reranker", questions, ground_truth_contexts, references)

## Сводная таблица

In [ ]:
df = pd.DataFrame([r["metrics"] for r in results.values()])
df = df.set_index("architecture")

print("=== Summary DataFrame ===")
df

## Сохранение результатов

In [ ]:
df.to_json(ARTIFACTS_DIR / "architecture_comparison.json", orient="index", indent=2)

detailed = {
    "config": {
        "n_samples": N_SAMPLES,
        "llm_model": OPENROUTER_MODEL,
        "embed_model": EMBED_MODEL,
        "reranker_model": RERANKER_MODEL,
        "top_k": TOP_K,
        "top_n": TOP_N,
    },
    "questions": questions,
    "references": references,
    "results": {k: {"predictions": v["predictions"], "latencies": v["latencies"]} for k, v in results.items()},
}

with open(ARTIFACTS_DIR / "detailed_results.json", "w", encoding="utf-8") as f:
    json.dump(detailed, f, ensure_ascii=False, indent=2, default=str)

print(f"Saved: {ARTIFACTS_DIR / 'architecture_comparison.json'}")
print(f"Saved: {ARTIFACTS_DIR / 'detailed_results.json'}")
print("\n=== Task 3 Complete ===")